In [1]:
from pathlib import Path
import json, sys, time

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from biorag.run_pipeline import run

QUESTION_FILE = ROOT / 'data/official/training11b.json'
CORPUS_FILE = ROOT / 'data/official/pubmed_abstracts.json'
OUTPUT_FILE = ROOT / 'artifacts/results/official_notebook_results.json'
assert QUESTION_FILE.exists() and CORPUS_FILE.exists(), 'Run the official ingestion notebook first.'
RUN_FULL = True
MAX_QUESTIONS = None
print('Official questions:', QUESTION_FILE)
print('Official corpus:', CORPUS_FILE)
print('Question limit: all 4,719 official questions')

Official questions: /home/anmol/BioRAG-Hybrid-Evaluation1/data/official/training11b.json
Official corpus: /home/anmol/BioRAG-Hybrid-Evaluation1/data/official/pubmed_abstracts.json
Question limit: all 4,719 official questions


In [2]:
start = time.time()
result = run(
    data_path=QUESTION_FILE,
    output_path=OUTPUT_FILE,
    corpus_path=CORPUS_FILE,
    lexical_weight=0.60,
    selector='hmm',
    max_questions=MAX_QUESTIONS,
)
print('Runtime seconds:', round(time.time() - start, 2))
print('Questions evaluated:', result['record_count'])
print('Corpus documents:', result['corpus_count'])
print('Aggregate metrics:', result['aggregate_metrics'])
print('Saved:', OUTPUT_FILE)

Runtime seconds: 5973.97
Questions evaluated: 4719
Corpus documents: 40160
Aggregate metrics: {'recall_at_3': 0.436163, 'mrr_at_3': 0.855725, 'summary_token_f1': 0.251309, 'rouge_l_f1': 0.198357}
Saved: /home/anmol/BioRAG-Hybrid-Evaluation1/artifacts/results/official_notebook_results.json


In [3]:
# Display one real BioASQ retrieval trace
record = result['records'][0]
print('Question:', record['question'])
print('Type:', record['question_type'])
print('Ranked documents:')
for item in record['ranked_documents']:
    print(item)
print('Selected evidence:')
for item in record['selected_evidence']:
    print(item)
print('Predicted extractive answer:', record['predicted_summary'])
print('Reference ideal answer:', record['reference_summary'])
print('Question-level metrics:', record['metrics'])

Question: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Type: summary
Ranked documents:
{'id': '15858239', 'title': '[The role of ret gene in the pathogenesis of Hirschsprung disease].', 'score': 0.98034, 'bm25_score': 25.095106, 'subword_cosine': 0.500076}
{'id': '6650562', 'title': 'Hirschsprung disease: etiologic implications of unsuccessful prenatal diagnosis.', 'score': 0.876194, 'bm25_score': 23.410031, 'subword_cosine': 0.416116}
{'id': '11106284', 'title': 'Genetics of Hirschsprung disease.', 'score': 0.870836, 'bm25_score': 24.483374, 'subword_cosine': 0.375329}
Selected evidence:
{'sentence': 'Hirschsprung disease is a congenital disorder with the incidence of 1 per 5000 live births, characterized by the absence of intestinal ganglion cells.', 'document_id': '15858239', 'state': 'SUPPORT', 'overlap': 0.6, 'position': 0}
{'sentence': 'In the etiology of Hirschsprung disease various genes play a role; these are: RET, EDNRB, GDNF, EDN3 and SOX10, NTN3, ECE1, 

In [4]:
# Validate result structure and metric ranges
assert result['status'] == 'executed'
assert result['corpus_count'] == 40160
for name, value in result['aggregate_metrics'].items():
    assert 0.0 <= value <= 1.0, (name, value)
assert all(record['ranked_documents'] for record in result['records'])
assert all(item['state'] in {'SUPPORT', 'CONTEXT'} for record in result['records'] for item in record['selected_evidence'])
print('Official retrieval output validation: PASS')
print('This is the full official-data run: PASS')

Official retrieval output validation: PASS
This is the full official-data run: PASS
